# TP 2 - Assistant RAG simple


Ce notebook construit un assistant RAG simple à partir de la base vectorielle V1 préparée dans `2_1_rag_db_preparation.ipynb`.

### 0.1. Objectif
- **TP 2_1** : Préparer la base vectorielle V1 (chunking par caractères, embeddings, indexation Chroma)
- **TP 2_2** : Créer un assistant RAG simple : vectoriser la question, récupérer les chunks les plus proches, générer une réponse ancrée
- **TP 2_3** : Préparer la base vectorielle V2 (chunking par en-têtes Markdown)
- **TP 2_4** : Créer un assistant RAG avec des méthodes avancées de retrieval (Multi-Query, HyDE, Reranking)

### 0.2. Documentation générale

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

[ChromaDB](https://docs.trychroma.com/)

In [ ]:
from pathlib import Path
import chromadb

from shared.config import ROOT_DIR
from shared.rag_utils import RAGChunk, rag_embed_text_batch, rag_embed_text_batch_local

# INFO : Choix entre local ou cloud (LLM)
from shared.llm_utils import (
    LLMRequest,
    run_llm, # Cloud
    #run_llm_local as run_llm, # Local
)

# INFO : Choix entre local ou cloud (embeddings)
rag_embed_fn = rag_embed_text_batch # Cloud
#rag_embed_fn = rag_embed_text_batch_local # Local

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
VECTOR_DB_DIR = DATA_DIR / "chroma_db_rag_v1"  # choisir entre chroma_db_rag_v1 et chroma_db_rag_v2 selon la base à tester

### 0.3. Use case principal
Votre objectif est de répondre à une question

In [ ]:
user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""


### 0.4. Récapitulatif des fonctions utilisées dans ce notebook

**Fournies**

- `run_llm` : fonction qui lit la configuration, envoie la requête au modèle et retourne un `LLMResponse`
- `LLMRequest` : classe qui représente les données d'entrée d'un appel LLM
- `LLMResponse` : classe qui représente les données de sortie utiles (texte final, tokens, données brutes)

--> Disponibles dans `shared/llm_utils.py`

- `rag_embed_text_batch` : fonction qui calcule les embeddings d'une liste de textes via l'API Google GenAI
- `rag_embed_all_chunks` : fonction qui calcule les embeddings de tous les chunks par batch
- `rag_index_chunks_chroma` : fonction qui indexe des chunks vectorisés dans une collection Chroma

--> Disponibles dans `shared/rag_utils.py`

**À coder dans ce notebook**

- `RAGAssistant` : classe qui implémente la recherche vectorielle sur base Chroma persistée

--> À implémenter ici, puis à copier dans `shared/rag_utils.py`

---
## 1. Créer l'assistant RAG

Chaque chunk indexé dans Chroma est représenté par un **vecteur d'embedding**.

Lors d'une recherche, **la requête est elle aussi vectorisée**, puis on calcule la similarité cosinus entre ce vecteur et tous les vecteurs de la base.

La **similarité cosinus** mesure l'angle entre deux vecteurs : une valeur proche de 1 indique que les deux textes sont alignés donc sémantiquement proches, même s'ils n'ont pas de mots en commun.

### 1.1. Coder la recherche vectorielle

In [ ]:
class RAGAssistant:
    """Assistant de recherche vectorielle sur base Chroma persistée"""

    def __init__(self, persist_dir: Path, top_k: int = 10, embed_fn=rag_embed_text_batch):
        self.persist_dir: Path = persist_dir
        self.top_k: int = top_k
        self.embed_fn = embed_fn
        if not self.persist_dir.exists():
            raise ValueError(f"persist_dir does not exist: {self.persist_dir}")
        client = chromadb.PersistentClient(path=str(self.persist_dir))
        self.collection = client.get_collection(name="chunks")

    def search(self, query: str, top_k: int | None = None) -> list[tuple[RAGChunk, float]]:
        """Rechercher les chunks les plus proches pour une requête

        Entrées
        - query : texte utilisateur
        - top_k : surcharge optionnelle du `top_k` par défaut

        Sortie
        - liste de paires `(RAGChunk, score)`
          - `RAGChunk` : chunk retrouvé (source, chunk_id, text)
          - `score` : score de similarité dérivé de la distance vectorielle
        """
        chunks_with_scores: list[tuple[RAGChunk, float]] = []
        # DOC (query) : https://docs.trychroma.com/docs/querying-collections/query-and-get
        ...
        return chunks_with_scores

### 1.2. Initialiser l'assistant

In [ ]:
rag_assistant = RAGAssistant(persist_dir=VECTOR_DB_DIR, embed_fn=rag_embed_fn)

---
## 2. Prompt Engineering

### 2.1. Rédiger le system prompt

Écrivez un **prompt système** similaire à celui du TP1, mais adapté à la méthode RAG.

Concrètement, il faut ajouter des instructions spécifiques à la gestion du contexte injecté, entre autres :
- **Interdire toute invention**, utiliser uniquement les informations fournies dans le contexte
- **Mentionner** clairement les **informations manquantes** pour répondre à la requête
- (optionnel) **Sourcer** chaque information

In [ ]:
system_prompt = """
Tu es un assistant de planification de voyage basé sur la méthode RAG.

### Règles :
- ...

### Contraintes :
- ...

### Format de réponse :
1) ...
"""

---
## 3. Récupérer le contexte (Retrieval)

Inspecter les chunks récupérés avant génération.

À vérifier :
- **Pertinence** : les chunks répondent-ils réellement à la requête ?
- **Couverture** : proviennent-ils de plusieurs sources utiles ?

### 3.1. Lancer la recherche de chunks

In [ ]:
top_chunks = ...

### 3.2. Afficher la répartition par source

Afficher combien de chunks ont été récupérés par source.

In [ ]:
...

---
## 4. Générer la réponse

### 4.1. Assembler le contexte et générer la réponse

Les chunks sont concaténés dans un bloc CONTEXTE, injectés dans le prompt système, puis envoyés au modèle.

Validation concrète de la sortie :
- chaque fait important doit être sourcé,
- aucune invention,
- les zones d'incertitude doivent être annoncées explicitement.

In [ ]:
top_chunks = rag_assistant.search(user_query)
context = ...
grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{context}"
run_result = await run_llm(...)

print(run_result.output)
print()
print(f"tokens : entrée={run_result.input_tokens} | sortie={run_result.output_tokens} | total={run_result.total_tokens}")

---
## Déplacer vers shared/

`RAGAssistant` est utilisée dans des TP ultérieurs.

Copiez la classe dans `shared/rag_utils.py` avant de passer à la suite.